# P04 · Notebook 11 · Pump sound in factory noise: the physics of the 8-microphone array

**The problem:** at −6 dB the factory is louder than the pump. Every published MIMII result, and our notebooks so far, listen with one microphone, although MIMII was recorded with eight.

**The physics:** the pump sits in one place, 50 cm from a ring of 8 microphones. Its sound reaches each microphone with a tiny, fixed delay; noise from elsewhere does not share those delays.
- Measure the pump's delays once, at installation: GCC-PHAT on normal training clips at 6 dB, the quietest level.
- Shift each channel back by its delay and add the eight. The pump adds up in step; the noise partly cancels.
- This is delay-and-sum beamforming. No training and no extra data.

**The test, at 6, 0 and −6 dB, with the same frozen BEATs model as notebook 1:**
- one microphone (CH0);
- the plain average of eight (AVG, more microphones without the physics);
- the beamformer (BF), steered with delays measured once at installation (6 dB, the quietest level);
- the same beamformer with delays re-estimated in the noise (BFs, reported only).

We also score every front end with the 2026 training-free method of Wilkinghoff, Yadav & Tan (RDP+GeM pooling and a normalised nearest neighbour).

**Decisions** S1–S3 are fixed in `results/mimii_beats/RULES_array_physics_noise_written_before_running.md`.

**Run:**
- Runtime › Change runtime type › **T4 GPU** (or better). Then Runtime › Run all.
- It downloads three zips of about 8 GB each and runs BEATs eleven times. On a T4 it takes about 1–1.5 hours.
- Each noise level is saved to Drive as soon as it is done. If Colab disconnects, run all again: finished levels are skipped.
- The BEATs checkpoint is read from Google Drive › MyDrive › `P04_BEATs`, where notebook 1 left it.

**Licences:** BEATs code and weights: MIT (Microsoft). MIMII: CC BY-SA 4.0 (Hitachi).

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, base64
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, WORK, CKPT_DIR = Path("/content/drive/MyDrive/P04_ArrayNoise"), Path("/content/work"), Path("/content/drive/MyDrive/P04_BEATs")
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_ArrayNoise")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", str(OUT)))
for p in (OUT, WORK, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: this will be very slow)")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()

In [ ]:
# 2 · The official BEATs code (the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 3 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 4 · Every number below was fixed in the rules before any of these clips was read
RULES = "results/mimii_beats/RULES_array_physics_noise_written_before_running.md"
LEVELS = {"6dB": ("https://zenodo.org/records/3384388/files/6_dB_pump.zip?download=1", "6_dB_pump.zip", "a09ba6060c10fc09cd4c8770213b0b9f"),
          "0dB": ("https://zenodo.org/records/3384388/files/0_dB_pump.zip?download=1", "0_dB_pump.zip", "488748295c3f60b25de07b58fe75b049"),
          "-6dB": ("https://zenodo.org/records/3384388/files/-6_dB_pump.zip?download=1", "-6_dB_pump.zip", "d20b783a0ff9c93d58f452f98c37b112")}
if TEST:
    LEVELS = json.loads(os.environ["P04_TEST_DATA"])
FRONTS = ["CH0", "AVG", "BF", "BFs"]       # BF: installation delays (6 dB); BFs: delays from the same noise level
NFFT, HOP, UPS, MAXLAG = 1024, 512, 16, 16          # GCC-PHAT: frame, hop, zero-padding factor, search ±16 samples
GAMMA, P_GEM = 16.0, 3.0                             # RDP+GeM for BEATs (Wilkinghoff, Yadav & Tan 2026)
ALPHAS = np.round(np.arange(-1.0, 2.0001, 0.02), 2)
SHOTS = [1, 3, 10]
N1_SPLIT_A = {"id_00": 0.9969, "id_02": 0.9965, "id_04": 0.9746, "id_06": 0.9936, "avg": 0.9904}   # notebook 1 / 3, CH0 at 6 dB, B1


def get_zip(level):
    url, name, want = LEVELS[level]
    for place in (OUT / name, WORK / name):
        if place.exists() and md5(place) == want:
            return place
    dest = WORK / name
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}")
    assert got == want, f"stop: {name} is not the published file"
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]

In [ ]:
# 5 · Physics front end: GCC-PHAT delays from training clips, then delay-and-sum on the whole clip
WIN = np.hanning(NFFT).astype(np.float32)


def cross_spectra(x):
    # x: (samples, 8). Sum over frames of conj(X_0)·X_m for m = 0..7 → (8, NFFT/2+1) complex
    n = (len(x) - NFFT) // HOP + 1
    idx = np.arange(NFFT)[None, :] + HOP * np.arange(n)[:, None]
    F = np.fft.rfft(x[idx] * WIN[None, :, None], axis=1)           # (frames, bins, 8)
    return np.einsum("fb,fbm->mb", np.conj(F[:, :, 0]), F)


def delays_from(G):
    # GCC-PHAT on summed cross-spectra G (8, bins) → delay of each channel against channel 0, in samples
    out = np.zeros(G.shape[0])
    for m in range(1, G.shape[0]):
        g = G[m] / np.maximum(np.abs(G[m]), 1e-12)
        r = np.fft.irfft(g, n=NFFT * UPS)
        K = MAXLAG * UPS
        rc = np.concatenate([r[-K:], r[:K + 1]])                     # lags −K..+K (in 1/UPS samples)
        out[m] = (np.argmax(rc) - K) / UPS
    return out


def delay_and_sum(X, tau, N):
    k = np.arange(X.shape[0])[:, None]
    return np.fft.irfft((X * np.exp(2j * np.pi * k * tau[None, :] / N)).mean(1), n=N).astype(np.float32)


def front_ends(x, tau, tau_same=None):
    # x: (samples, 8) → mono signals; BF advances channel m by tau[m] samples, then averages
    N = len(x)
    X = np.fft.rfft(x, axis=0)
    out = {"CH0": x[:, 0].copy(), "AVG": x.mean(1).astype(np.float32), "BF": delay_and_sum(X, tau, N)}
    if tau_same is not None:
        out["BFs"] = delay_and_sum(X, tau_same, N)
    return out


# smoke test: a broadband source arriving with known fractional delays, plus independent noise 6 dB louder
rng = np.random.default_rng(0)
true = np.r_[0.0, rng.uniform(-4, 4, 7)]
s = rng.standard_normal(160000).astype(np.float32)
S = np.fft.rfft(s); kk = np.arange(len(S))
xs = np.stack([np.fft.irfft(S * np.exp(-2j * np.pi * kk * t / 160000), n=160000) for t in true], 1).astype(np.float32)
xn = xs + 2.0 * rng.standard_normal(xs.shape).astype(np.float32)
est = delays_from(cross_spectra(xn))
fe = front_ends(xn, est)
gain = lambda y: 10 * np.log10(np.var(s) / np.var(y - s * (np.dot(y, s) / np.dot(s, s))))
print("delays true", np.round(true, 2).tolist(), "\nestimated  ", np.round(est, 2).tolist())
print(f"SNR after each front end (dB): CH0 {gain(fe['CH0']):.1f} · AVG {gain(fe['AVG']):.1f} · BF {gain(fe['BF']):.1f}")
assert np.max(np.abs(est - true)) < 0.25, "smoke test: GCC-PHAT does not recover the delays"
assert gain(fe["BF"]) > gain(fe["CH0"]) + 6, "smoke test: delay-and-sum does not gain the expected ~9 dB"

In [ ]:
# 6 · BEATs on a batch of mono clips: float32 filter banks and encoder (as notebook 1), two poolings of the last layer
@torch.no_grad()
def embed(batch):
    x = torch.from_numpy(np.stack(batch)).to(DEVICE)
    h, _ = model.extract_features(x)                                # (B, T, 768), the model's own output
    mean = h.mean(1)
    d = (h - mean[:, None, :]).norm(dim=2)                           # deviation of each frame from the clip mean
    w = (1 + d / d.max(1, keepdim=True).values) ** GAMMA
    w = w / w.sum(1, keepdim=True)
    rg = (w[:, :, None] * h.clamp(min=0) ** P_GEM).sum(1) ** (1 / P_GEM)
    return mean.float().cpu().numpy(), rg.float().cpu().numpy()


m1, r1 = embed([np.random.default_rng(1).uniform(-0.1, 0.1, 160000).astype(np.float32)] * 2)
print("embedding shapes:", m1.shape, r1.shape, "· two identical clips give identical rows:", bool(np.allclose(m1[0], m1[1])))

In [ ]:
# 7 · For each noise level: delays from training clips, then BEATs on every front end for every clip (saved to Drive)
FIRST = list(LEVELS)[0]                                              # 6 dB: the installation measurement


def run_level(level, batch_size=16):
    out = OUT / f"array_{level}_pump.npz"
    if out.exists():
        print(level, "already done:", out.name); return
    zpath = get_zip(level)
    zf = zipfile.ZipFile(zpath)
    names = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
    pump, label = parse(names)
    pumps = sorted(set(pump.tolist()))
    t0 = time.time()
    tau, tau_half = {}, {}
    for p in pumps:                                                  # pass 1: training clips only
        tr, _ = split_a(pump, label, names, p)
        tr = tr[np.argsort(names[tr])]
        Gs = [0, 0]
        for j, i in enumerate(tr):
            x, sr = read_all(zf.read(names[i]))
            assert sr == 16000 and x.shape[1] == 8, (names[i], sr, x.shape)
            Gs[j % 2] = Gs[j % 2] + cross_spectra(x)
        tau[p] = delays_from(Gs[0] + Gs[1])
        tau_half[p] = (delays_from(Gs[0]), delays_from(Gs[1]))
        print(f"  {level} {p}: delays (samples) {np.round(tau[p], 2).tolist()} · halves differ by at most "
              f"{np.max(np.abs(tau_half[p][0] - tau_half[p][1])):.2f} · {time.time() - t0:.0f} s", flush=True)
    if level == FIRST:
        tau_bf, fronts = tau, [f for f in FRONTS if f != "BFs"]       # at installation, BF and BFs are the same thing
    else:
        tau_bf = {p: np.array(v) for p, v in json.loads(str(np.load(OUT / f"array_{FIRST}_pump.npz")["tau"])).items()}
        fronts = FRONTS
    EMB = {f: {"mean": [], "rdpgem": []} for f in fronts}
    buf = {f: [] for f in fronts}

    def flush(f):
        a, b = embed(buf[f]); EMB[f]["mean"].append(a); EMB[f]["rdpgem"].append(b); buf[f].clear()

    for i, n in enumerate(names):                                    # pass 2: every clip
        x, sr = read_all(zf.read(n))
        fe = front_ends(x, tau_bf[pump[i]], None if level == FIRST else tau[pump[i]])
        for f in fronts:
            buf[f].append(fe[f])
            if len(buf[f]) == batch_size:
                flush(f)
        if i % 500 == 0:
            print(f"  {level}: {i}/{len(names)} clips · {time.time() - t0:.0f} s", flush=True)
    for f in fronts:
        if buf[f]:
            flush(f)
    if level == FIRST:
        EMB["BFs"] = EMB["BF"]
    meta = {"level": level, "zip": LEVELS[level][1], "zip_md5": LEVELS[level][2], "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
            "clips": int(len(names)), "runtime_s": round(time.time() - t0, 1), "device": DEVICE,
            "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "torch": torch.__version__, "test_run": TEST,
            "date": time.strftime("%Y-%m-%d %H:%M")}
    np.savez_compressed(out, fname=names, pump=pump, label=label,
                        **{f"{f}_{k}": np.concatenate(EMB[f][k]) for f in FRONTS for k in ("mean", "rdpgem")},
                        tau=json.dumps({p: v.tolist() for p, v in tau.items()}),
                        tau_half=json.dumps({p: [h[0].tolist(), h[1].tolist()] for p, h in tau_half.items()}),
                        meta=json.dumps(meta))
    print(f"{level}: saved {out.name} · {len(names)} clips · {meta['runtime_s']:.0f} s")
    if IN_COLAB and zpath.parent == WORK:
        zpath.unlink()                                               # free the disk for the next level


for level in LEVELS:
    run_level(level)

In [ ]:
# 8 · Scores: B1 (notebook 1's scorer) and W26 (Wilkinghoff et al. 2026), for every level, front end and split
def b1(R, X):
    R = R / np.linalg.norm(R, axis=1, keepdims=True); X = X / np.linalg.norm(X, axis=1, keepdims=True)
    return 1.0 - (X @ R.T).max(1)


def pdist(A, B):
    d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(d2, 1e-24))


def w26(R, X):
    DRR = pdist(R, R); np.fill_diagonal(DRR, np.inf)
    ld1 = np.log(DRR.min(1))                                         # log distance of each reference to its nearest other
    LRR = np.log(DRR)                                                # leave-one-out: the diagonal stays +inf
    var = [np.var((LRR - a * ld1[None, :]).min(1)) for a in ALPHAS]
    a = float(ALPHAS[int(np.argmin(var))])
    return (np.log(pdist(X, R)) - a * ld1[None, :]).min(1), a


def auc(y, s):
    return round(float(roc_auc_score(y, s)), 4)


RES, ALPHA = {}, {}
for level in LEVELS:
    D = np.load(OUT / f"array_{level}_pump.npz", allow_pickle=False)
    names, pump, label = D["fname"], D["pump"], D["label"]
    pumps = sorted(set(pump.tolist()))
    A = {p: split_a(pump, label, names, p) for p in pumps}
    for f in FRONTS:
        for sc, key in (("B1", "mean"), ("W26", "rdpgem")):
            E = D[f"{f}_{key}"].astype(np.float64)
            r = {}
            for p in pumps:
                tr, te = A[p]
                fleet = np.concatenate([A[q][0] for q in pumps if q != p])
                y = label[te]
                score = (lambda R, X: b1(R, X)) if sc == "B1" else (lambda R, X: w26(R, X)[0])
                r.setdefault("A", {})[p] = auc(y, score(E[tr], E[te]))
                r.setdefault("B", {})[p] = auc(y, score(E[fleet], E[te]))
                for n in SHOTS:
                    r.setdefault(f"B+{n}", {})[p] = auc(y, score(E[np.r_[fleet, tr[:n]]], E[te]))
            for k in r:
                r[k]["avg"] = round(float(np.mean([r[k][p] for p in pumps])), 4)
            RES[(level, f, sc)] = r
    print(f"{level} scored")

print(f"\n{'level':6s} {'front':5s} {'scorer':6s} " + " ".join(f"{k:>7s}" for k in ["A", "B", "B+1", "B+3", "B+10"]))
for (level, f, sc), r in RES.items():
    print(f"{level:6s} {f:5s} {sc:6s} " + " ".join(f"{r[k]['avg']:7.4f}" for k in ["A", "B", "B+1", "B+3", "B+10"]))

In [ ]:
# 9 · Gates and decisions G0, G1, S1–S3, exactly as written in the rules
A6, A0, Am6 = (RES[(lv, "CH0", "B1")]["A"]["avg"] for lv in ("6dB", "0dB", "-6dB"))
G0 = abs(A6 - N1_SPLIT_A["avg"]) <= 0.005
print(f"G0 · CH0 at 6 dB reproduces notebook 1: {A6} vs {N1_SPLIT_A['avg']} →", "pass" if G0 else "FAIL",
      "· per pump", {p: RES[('6dB', 'CH0', 'B1')]['A'][p] for p in N1_SPLIT_A if p != 'avg'})
if not TEST:
    assert G0, "stop: the pipeline does not reproduce notebook 1"
G1 = Am6 <= A6 - 0.02
loss = A6 - Am6
bf, avg_ = RES[("-6dB", "BF", "B1")]["A"]["avg"], RES[("-6dB", "AVG", "B1")]["A"]["avg"]
R_rec = (bf - Am6) / loss if loss > 0 else float("nan")
b10_bf, b10_ch0 = RES[("-6dB", "BF", "B1")]["B+10"]["avg"], RES[("-6dB", "CH0", "B1")]["B+10"]["avg"]
S = {"S1": (G1 and R_rec >= 0.5, G1, f"noise loss {loss:.4f} (6 dB {A6} → −6 dB {Am6}); BF at −6 dB {bf}; recovered share R = {R_rec:.2f}"),
     "S2": (bf - avg_ >= 0.01, True, f"BF {bf} vs AVG {avg_} at −6 dB → {bf - avg_:+.4f}"),
     "S3": (b10_bf - b10_ch0 >= 0.02, True, f"B+10 at −6 dB: BF {b10_bf} vs CH0 {b10_ch0} → {b10_bf - b10_ch0:+.4f}")}
TXT = {"S1": ("aligning the eight microphones wins back at least half of what the noise took", "it wins back less than half"),
       "S2": ("and it is the alignment that matters, not only having more microphones", "the plain average does about as well"),
       "S3": ("a new pump commissioned with 10 clips also gains in noise", "no clear gain for a new pump with 10 clips")}
VERDICT = {k: ((TXT[k][0] if ok else TXT[k][1]) if dec else "not decidable: the noise does not hurt this pipeline") for k, (ok, dec, _) in S.items()}
print("G1 · the noise hurts (≥ 0.02):", "yes" if G1 else "no")
for k, (ok, dec, det) in S.items():
    print(f"{k} → {VERDICT[k]}  ·  {det}")

In [ ]:
# 10 · The delays the array found, per pump and level (physics check: the pump does not move when the noise changes)
TAU = {}
for level in LEVELS:
    D = np.load(OUT / f"array_{level}_pump.npz", allow_pickle=False)
    TAU[level] = {"tau": json.loads(str(D["tau"])), "half": json.loads(str(D["tau_half"]))}
for p in sorted(TAU["6dB"]["tau"]):
    rows = {lv: np.array(TAU[lv]["tau"][p]) for lv in LEVELS}
    spread = max(np.max(np.abs(rows[a] - rows[b])) for a in rows for b in rows)
    halves = {lv: float(np.max(np.abs(np.array(TAU[lv]["half"][p][0]) - np.array(TAU[lv]["half"][p][1])))) for lv in LEVELS}
    print(f"{p}: 6 dB delays {np.round(rows['6dB'], 2).tolist()} samples · largest change across levels {spread:.2f} · "
          f"halves differ by at most {halves}")

In [ ]:
# 11 · Save, and three pictures
RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "checkpoint_sha256": CKPT_SHA,
          "results": {f"{lv}|{f}|{sc}": r for (lv, f, sc), r in RES.items()}, "delays_samples": TAU,
          "G0": {"CH0_6dB_A": A6, "notebook1": N1_SPLIT_A["avg"], "pass": bool(G0)}, "G1": bool(G1),
          "decisions": {k: {"pass": bool(ok), "decidable": bool(dec), "detail": det} for k, (ok, dec, det) in S.items()}, "verdicts": VERDICT}
name = "array_noise.json" if not (OUT / "array_noise.json").exists() else "array_noise_rerun.json"
(OUT / name).write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
print("saved", name)

import matplotlib.pyplot as plt
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COL = {"CH0": "#8d8d8d", "AVG": "#eb6834", "BF": "#2a78d6", "BFs": "#8fb8e8"}
LAB = {"CH0": "one microphone (CH0)", "AVG": "average of 8, no alignment", "BF": "8 microphones aligned (physics, installation delays)",
       "BFs": "aligned, delays re-estimated in the noise"}
lv = list(LEVELS); xs = [6, 0, -6]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for ax, split, title in ((axes[0], "A", "pump the model knows (Split A)"), (axes[1], "B+10", "new pump, 10 normal clips (B+10)")):
    for f in FRONTS:
        ax.plot(xs, [RES[(l, f, "B1")][split]["avg"] for l in lv], "-o" if f != "BFs" else "--o", color=COL[f], label=LAB[f], lw=2)
    ax.set_xticks(xs, ["6 dB", "0 dB", "−6 dB"]); ax.invert_xaxis(); ax.set_title(title, loc="left", color=INK, fontsize=10.5)
    ax.set_xlabel("signal-to-noise ratio (factory noise grows →)", color=MUTED); ax.grid(True, color=GRID, lw=0.6)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(colors=MUTED)
axes[0].set_ylabel("average AUC over 4 pumps", color=MUTED); axes[1].legend(frameon=False, loc="lower left", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "array_noise_auc.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.3), sharey=True)
for ax, p in zip(axes, sorted(TAU["6dB"]["tau"])):
    for i, l in enumerate(lv):
        ax.plot(range(8), np.array(TAU[l]["tau"][p]) / 16000 * 1e6, "o-", ms=4, lw=1, label=l, color=["#173f7a", "#4f8fd6", "#8fb8e8"][i])
    ax.set_title(p, loc="left", color=INK, fontsize=10); ax.set_xlabel("microphone", color=MUTED); ax.grid(True, color=GRID, lw=0.6)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("delay vs microphone 0 (µs)", color=MUTED); axes[-1].legend(frameon=False, fontsize=8.5)
fig.suptitle("Where the array hears each pump, estimated separately at each noise level", x=0.01, ha="left", color=INK, fontsize=10.5)
fig.tight_layout(); fig.savefig(OUT / "array_noise_delays.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.6))
pumps = sorted(TAU["6dB"]["tau"]); x = np.arange(len(pumps)); w = 0.26
w = 0.2
for i, f in enumerate(FRONTS):
    ax.bar(x + (i - 1.5) * w, [RES[("-6dB", f, "B1")]["A"][p] for p in pumps], w, color=COL[f], label=LAB[f])
ax.set_xticks(x, pumps); ax.set_ylim(0.4, 1.02); ax.set_ylabel("AUC at −6 dB (Split A)", color=MUTED)
ax.grid(True, axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.legend(frameon=False, fontsize=8.5, loc="lower right")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout(); fig.savefig(OUT / "array_noise_per_pump.png", dpi=150); plt.show()

In [ ]:
# 12 · Put the results in one zip and download it (the large embedding files stay in Drive)
names = [n for n in ("array_noise.json", "array_noise_rerun.json", "array_noise_auc.png", "array_noise_delays.png", "array_noise_per_pump.png")
         if (OUT / n).exists()]
ZIP = OUT / "p04_n11_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for n in names:
        z.write(OUT / n, n)
print("zipped:", names, "→", ZIP)
if IN_COLAB:
    from google.colab import files as _f
    _f.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n11_results.zip`), or paste the printed output of cells 7–10 in the chat.